# ¿Qué hogares presentan inseguridad alimentaria?

**Taller: Análisis de datos con IA y aprendizaje automático**
Dr. Juan Carlos De la Cruz Maldonado · FCAV-UAT ·

Este cuaderno recorre, de principio a fin, un análisis con aprendizaje automático sobre la
**Encuesta Nacional de Salud y Nutrición (ENSANUT) Continua 2025** del Instituto Nacional de
Salud Pública. No hay que escribir código: solo ejecutarlo y leer lo que muestra.

**Cómo usarlo.** En el menú, *Entorno de ejecución → Ejecutar todo*. Cada celda tiene un título
que dice qué hace; el código está plegado (el ícono ‹› lo muestra si quieren verlo).

**La pregunta.** ¿Qué hogares tienen más probabilidad de presentar inseguridad alimentaria moderada
o severa, y qué perfiles conviene priorizar?

| Pieza | En este caso |
|---|---|
| Población | Hogares de México captados por la ENSANUT 2025 |
| Resultado a explicar | Presenta o no inseguridad alimentaria moderada o severa (escala ELCSA) |
| Factores | Ámbito, jefatura, integrantes, menores, nivel socioeconómico, programas sociales, escolaridad |
| Decisión que informa | Dónde y para qué hogares priorizar comedores u orientación alimentaria |

In [ ]:
#@title 1. Cargar la base de hogares
# Las bibliotecas ya vienen instaladas en Colab: no hay que instalar nada.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

URL_DATOS = "https://raw.githubusercontent.com/USUARIO/REPOSITORIO/main/datos/ensanut2025_hogares.csv"
try:
    hogares = pd.read_csv(URL_DATOS)
except Exception:
    # Copia local (solo para pruebas fuera de Colab)
    hogares = pd.read_csv("../datos/ensanut2025_hogares.csv")

# Paleta institucional (estilos/estilos.txt)
AZUL, NARANJA, GRIS, OBSCURO = "#003d5c", "#d05f27", "#CFCDC9", "#55565b"

n_encuestados = len(hogares)
n_representados = hogares["factor_expansion"].sum()
print(f"La base tiene {n_encuestados:,} hogares encuestados.")
print(f"Con el factor de expansión representan a {n_representados/1e6:.1f} millones de hogares del país.")
print()
print("Así se ve un hogar en la base (cada renglón es un hogar):")
columnas_ver = ["entidad", "ambito", "jefatura", "integrantes", "hay_menores",
                "nivel_socioeconomico", "recibe_programa_social", "inseguridad_alimentaria"]
display(hogares[columnas_ver].sample(5, random_state=1).reset_index(drop=True))

**Factor de expansión, en lenguaje llano.** La ENSANUT no visita a todos los hogares del país,
sino a una muestra. Cada hogar encuestado "habla por" cientos o miles de hogares parecidos. Por
eso todos los porcentajes de este cuaderno se calculan con ese peso: así describen al país y no
solo a los encuestados.

In [ ]:
#@title 2. ¿Cuántos hogares presentan inseguridad alimentaria?
def porcentaje_ponderado(df, columna, valor="Sí"):
    """Porcentaje de hogares (ponderado) donde columna == valor."""
    w = df["factor_expansion"]
    return 100 * w[df[columna] == valor].sum() / w.sum()

w = hogares["factor_expansion"]
niveles = ["Seguridad", "Leve", "Moderada", "Severa"]
nacional = hogares.groupby("inseguridad_alimentaria")["factor_expansion"].sum() / w.sum() * 100
nacional = nacional.reindex(niveles)

print("De cada 100 hogares del país, según la escala ELCSA de la ENSANUT 2025:")
for nivel, p in nacional.items():
    etiqueta = {"Seguridad": "tienen seguridad alimentaria",
                "Leve": "presentan inseguridad alimentaria leve",
                "Moderada": "presentan inseguridad alimentaria moderada",
                "Severa": "presentan inseguridad alimentaria severa"}[nivel]
    print(f"  {p:5.1f}  {etiqueta}")
p_mod_sev = porcentaje_ponderado(hogares, "inseguridad_moderada_o_severa")
print(f"\nEs decir, {p_mod_sev:.1f} de cada 100 hogares presentan inseguridad MODERADA O SEVERA. Ese es el resultado que el modelo intentará explicar.")

print("\n¿Y por tipo de hogar? (de cada 100 hogares de cada grupo)")
tablas = {}
for factor, titulo in [("ambito", "Ámbito"), ("jefatura", "Jefatura del hogar"),
                       ("hay_menores", "Hay menores de 18 años"),
                       ("nivel_socioeconomico", "Nivel socioeconómico"),
                       ("recibe_programa_social", "Recibe algún programa social"),
                       ("escolaridad_jefe", "Escolaridad del jefe o jefa")]:
    t = hogares.groupby(factor).apply(lambda g: porcentaje_ponderado(g, "inseguridad_moderada_o_severa"))
    t = t.round(1).rename("Con inseguridad moderada o severa (de cada 100)")
    tablas[factor] = t
    print(f"\n{titulo}")
    for categoria, p in t.sort_values(ascending=False).items():
        print(f"  {str(categoria):<22} {p:5.1f}")

**Validar antes de interpretar.** El porcentaje nacional de la celda anterior se compara con el
que publica el INSP para esta edición. Si no coincidiera, algo falló en la preparación de la base
y no tendría sentido seguir. (La clasificación ELCSA de esta base reproduce exactamente las
cifras del informe nacional ENSANUT Continua 2023 del INSP.)

**Preguntas para el grupo.** ¿Coincide con lo que ven en campo? ¿Qué les sorprende? ¿Qué
hogares creen que la encuesta no alcanza a captar?

In [ ]:
#@title 3. Gráfica: ¿a qué hogares les pasa más?
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.2))
for eje, (factor, titulo) in zip(ejes, [("nivel_socioeconomico", "Nivel socioeconómico"),
                                        ("ambito", "Ámbito"), ("jefatura", "Jefatura")]):
    t = tablas[factor].sort_values()
    colores = [NARANJA if v == t.max() else AZUL for v in t.values]
    barras = eje.barh(t.index.astype(str), t.values, color=colores)
    for b, v in zip(barras, t.values):
        eje.text(v + 0.4, b.get_y() + b.get_height() / 2, f"{v:.0f}", va="center", color=OBSCURO, fontsize=11)
    eje.set_title(titulo, color=AZUL, fontsize=13, loc="left", fontweight="bold")
    eje.set_xlim(0, max(t.values) * 1.25)
    eje.set_xlabel("De cada 100 hogares, con inseguridad moderada o severa", color=OBSCURO, fontsize=9)
    for lado in ["top", "right"]: eje.spines[lado].set_visible(False)
    eje.tick_params(colors=OBSCURO)
plt.suptitle("Inseguridad alimentaria moderada o severa por tipo de hogar (ENSANUT 2025)",
             color=OBSCURO, fontsize=12, x=0.01, ha="left")
plt.tight_layout()
plt.show()

---
## El árbol de decisión como tipología

Hasta aquí solo describimos. Ahora el modelo **aprende** de los datos: busca por sí mismo qué
preguntas sucesivas separan mejor a los hogares con inseguridad alimentaria de los que no.

- Se apartan **70 de cada 100 hogares para aprender** y **30 para comprobar**, como formarse
  con expedientes cerrados y después probar el criterio con expedientes nuevos.
- El árbol se dibuja completo. Cada camino desde arriba hasta una hoja es un **perfil de hogar**,
  igual que una categoría de una tipología cualitativa.
- La casilla **incluir_factor_trampa** agrega una pregunta de la propia escala ELCSA. Déjenla
  apagada la primera vez; después la encienden y vuelven a ejecutar desde aquí.

In [ ]:
#@title 4. Construir y dibujar el árbol
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split

incluir_factor_trampa = False  #@param {type:"boolean"}
niveles_del_arbol = 3  #@param {type:"slider", min:2, max:4, step:1}

# Factores en forma numérica para que el árbol pueda hacer cortes legibles
escolaridad_orden = ["sin escolaridad", "primaria o menos", "secundaria", "preparatoria", "licenciatura o más"]
X = pd.DataFrame({
    "Hogar rural": (hogares["ambito"] == "Rural").astype(int),
    "Jefatura femenina": (hogares["jefatura"] == "Femenina").astype(int),
    "Integrantes": hogares["integrantes"],
    "Hay menores de 18": (hogares["hay_menores"] == "Sí").astype(int),
    "Nivel socioeconómico bajo": hogares["nivel_socioeconomico"].str[0].astype(int).le(2).astype(int),
    "Recibe programa social": (hogares["recibe_programa_social"] == "Sí").astype(int),
    "Escolaridad del jefe": hogares["escolaridad_jefe"].str.lower().map({e: i for i, e in enumerate(escolaridad_orden)}),
})
if incluir_factor_trampa:
    X["TRAMPA: un adulto dejó de comer"] = (hogares["adulto_dejo_de_comer"] == "Sí").astype(int)
y = (hogares["inseguridad_moderada_o_severa"] == "Sí").astype(int)
peso = hogares["factor_expansion"]

X_aprender, X_comprobar, y_aprender, y_comprobar, w_aprender, w_comprobar = train_test_split(
    X, y, peso, test_size=0.30, random_state=2025, stratify=y)
print(f"Hogares para aprender: {len(X_aprender):,}   Hogares para comprobar: {len(X_comprobar):,}")

arbol = DecisionTreeClassifier(max_depth=niveles_del_arbol, min_samples_leaf=120,
                               class_weight="balanced", random_state=2025)
arbol.fit(X_aprender, y_aprender, sample_weight=w_aprender)

# ---- Traducción de cada pregunta del árbol a una frase ----
BINARIAS = {
    "Hogar rural": ("el hogar no es rural", "el hogar es rural"),
    "Jefatura femenina": ("la jefatura es masculina", "la jefatura es femenina"),
    "Hay menores de 18": ("no hay menores de 18", "hay menores de 18"),
    "Nivel socioeconómico bajo": ("el nivel socioeconómico es medio o alto (3 a 5)", "el nivel socioeconómico es bajo (1 o 2)"),
    "Recibe programa social": ("no recibe programas sociales", "recibe algún programa social"),
    "TRAMPA: un adulto dejó de comer": ("ningún adulto dejó de comer por falta de dinero", "algún adulto dejó de comer por falta de dinero"),
}
def frase(nombre, umbral, cumple):
    # Frase para la rama donde 'nombre <= umbral' se cumple (izquierda) o no (derecha)
    if nombre in BINARIAS:
        return BINARIAS[nombre][0 if cumple else 1]
    n = int(umbral)
    if nombre == "Integrantes":
        return f"tiene hasta {n} integrantes" if cumple else f"tiene más de {n} integrantes"
    if nombre == "Escolaridad del jefe":
        return (f"el jefe o jefa tiene {escolaridad_orden[n]} como máximo" if cumple
                else f"el jefe o jefa tiene {escolaridad_orden[n + 1]} o más").replace("o más o más", "o más").replace("tiene sin escolaridad como máximo", "no tiene escolaridad").replace("primaria o menos como máximo", "primaria o menos")
    return f"{nombre} {'<=' if cumple else '>'} {umbral:.1f}"

# Proporción REAL (ponderada con el factor de expansión) de hogares con inseguridad en cada nodo,
# calculada con los hogares de aprendizaje
t = arbol.tree_
nodo_de_cada_hogar = arbol.decision_path(X_aprender).toarray()
peso_nodo = nodo_de_cada_hogar.T @ w_aprender.values
positivos_nodo = nodo_de_cada_hogar.T @ (w_aprender.values * y_aprender.values)
p_nodo = 100 * positivos_nodo / peso_nodo
parte_nodo = 100 * peso_nodo / peso_nodo[0]

fig, eje = plt.subplots(figsize=(26, 9.5))
cajas = plot_tree(arbol, feature_names=list(X.columns), filled=False, rounded=True,
                  impurity=False, proportion=True, fontsize=10, ax=eje, label="none")
# Reescribe cada caja en español, con la proporción real y un color según esa proporción
cajas_nodo = [c for c in cajas if c.get_text().strip() not in ("True", "False")]  # quita las etiquetas de las flechas
for nodo, caja in enumerate(cajas_nodo):
    texto = ""
    if t.children_left[nodo] != -1:
        texto = f"¿{frase(list(X.columns)[t.feature[nodo]], t.threshold[nodo], True).capitalize()}?\n"
    texto += f"{parte_nodo[nodo]:.0f} de cada 100 hogares\n{p_nodo[nodo]:.0f} de cada 100 con inseguridad"
    caja.set_text(texto)
    k = min(p_nodo[nodo] / 40, 1.0)
    color = (1 - k * (1 - 0xd0/255), 1 - k * (1 - 0x5f/255), 1 - k * (1 - 0x27/255))
    caja.set_bbox(dict(boxstyle="round", fc=color, ec=OBSCURO))
for texto in eje.texts:
    if texto.get_text().strip() == "True": texto.set_text("sí  ")
    if texto.get_text().strip() == "False": texto.set_text("  no")
titulo = "Árbol de decisión: inseguridad alimentaria moderada o severa"
if incluir_factor_trampa: titulo += "   (con el factor trampa)"
eje.set_title(titulo, color=AZUL, fontsize=14, loc="left", fontweight="bold")
plt.show()
print("Cómo leerlo: cada caja hace una pregunta; a la izquierda van los hogares que responden 'sí' y a la derecha los que responden 'no'.")
print("Entre más naranja la caja, mayor es la proporción de hogares con inseguridad alimentaria en ese grupo.")

In [ ]:
#@title 5. Las ramas, leídas como perfiles de hogar
ORDINALES = {"Integrantes", "Escolaridad del jefe"}
def perfil_en_frases(condiciones):
    # Junta las condiciones de una rama en una sola frase por factor (sin repeticiones)
    limites = {}; frases = []
    for nombre, umbral, cumple in condiciones:
        if nombre in ORDINALES:
            lo, hi = limites.get(nombre, (None, None))
            if cumple: hi = int(umbral) if hi is None else min(hi, int(umbral))
            else: lo = int(umbral) + 1 if lo is None else max(lo, int(umbral) + 1)
            limites[nombre] = (lo, hi)
        else:
            frases.append(frase(nombre, umbral, cumple))
    for nombre, (lo, hi) in limites.items():
        if nombre == "Integrantes":
            if lo is not None and hi is not None: frases.append(f"tiene entre {lo} y {hi} integrantes" if lo != hi else f"tiene {lo} integrantes")
            elif hi is not None: frases.append(f"tiene hasta {hi} integrantes")
            else: frases.append(f"tiene {lo} integrantes o más")
        else:
            e = escolaridad_orden
            if lo is not None and hi is not None: frases.append(f"el jefe o jefa tiene {e[lo]}" if lo == hi else f"el jefe o jefa tiene entre {e[lo]} y {e[hi]}")
            elif hi is not None: frases.append(f"el jefe o jefa tiene {e[hi]} como máximo")
            else: frases.append((f"el jefe o jefa tiene {e[lo]} o más").replace("o más o más", "o más"))
    arreglos = {"primaria o menos o más": "primaria o más", "tiene sin escolaridad como máximo": "no tiene escolaridad",
                "primaria o menos como máximo": "primaria o menos", "sin escolaridad o más": "cualquier escolaridad"}
    def limpiar(f):
        for a, b in arreglos.items(): f = f.replace(a, b)
        return f
    return [limpiar(f) for f in frases]

nombres = list(X.columns)
hojas = []
def recorrer(nodo, condiciones):
    if t.children_left[nodo] == -1:
        hojas.append((nodo, condiciones)); return
    f, u = nombres[t.feature[nodo]], t.threshold[nodo]
    recorrer(t.children_left[nodo], condiciones + [(f, u, True)])
    recorrer(t.children_right[nodo], condiciones + [(f, u, False)])
recorrer(0, [])

# Qué pasa en cada perfil, medido en los hogares de COMPROBACIÓN (los que el árbol no vio)
hoja_de_cada_hogar = arbol.apply(X_comprobar)
filas = []
for nodo, condiciones in hojas:
    en_hoja = hoja_de_cada_hogar == nodo
    w = w_comprobar[en_hoja]
    if w.sum() == 0: continue
    p = 100 * w[y_comprobar[en_hoja] == 1].sum() / w.sum()
    filas.append((p, int(en_hoja.sum()), perfil_en_frases(condiciones)))
filas.sort(reverse=True)

print("Perfiles de hogar que encontró el árbol, del más al menos afectado:\n")
for i, (p, n, frases) in enumerate(filas, 1):
    de_cada_10 = "menos de 1" if p < 5 else f"{p/10:.0f}"
    verbo = "presenta" if de_cada_10 in ("menos de 1", "1") else "presentan"
    print(f"Perfil {i}. Si " + ", ".join(frases) + ":")
    print(f"   de cada 10 hogares así, {de_cada_10} {verbo} inseguridad alimentaria moderada o severa "
          f"({p:.0f} de cada 100; {n} hogares de comprobación).\n")

---
## ¿Cuánto acierta? La matriz de confusión

Se mide en los **30 de cada 100 hogares de comprobación**, que el modelo nunca vio. La matriz
cruza lo que dice la realidad (la escala ELCSA) con lo que dijo el modelo. De sus cuatro casillas
salen todas las métricas que aparecen en los artículos: exactitud (*accuracy*), sensibilidad
(*recall*), precisión (*precision*), especificidad, F1 y área bajo la curva (*AUC*).

In [ ]:
#@title 6. Matriz de confusión y métricas: ¿cuánto acierta el modelo?
from sklearn.metrics import confusion_matrix, roc_auc_score

prediccion = arbol.predict(X_comprobar)
vn, fp, fn, vp = confusion_matrix(y_comprobar, prediccion).ravel()
total = vp + vn + fp + fn

print(f"Medido en los {total:,} hogares de comprobación, que el árbol nunca vio.\n")
print("MATRIZ DE CONFUSIÓN")
matriz = pd.DataFrame(
    [[vp, fn, vp + fn], [fp, vn, fp + vn], [vp + fp, fn + vn, total]],
    index=["Realidad: SÍ presenta inseguridad", "Realidad: NO presenta inseguridad", "Total"],
    columns=["El modelo dijo SÍ", "El modelo dijo NO", "Total"])
display(matriz)
print(f"  Verdaderos positivos  (VP) {vp:>5,}  presentan inseguridad y el modelo los señaló.")
print(f"  Falsos negativos      (FN) {fn:>5,}  presentan inseguridad y el modelo NO los señaló: se le escaparon.")
print(f"  Falsos positivos      (FP) {fp:>5,}  no presentan inseguridad y el modelo los señaló: señalados por error.")
print(f"  Verdaderos negativos  (VN) {vn:>5,}  no presentan inseguridad y el modelo no los señaló.\n")

exactitud     = (vp + vn) / total
sensibilidad  = vp / (vp + fn)
precision     = vp / (vp + fp)
especificidad = vn / (vn + fp)
f1            = 2 * precision * sensibilidad / (precision + sensibilidad)
exact_balanceada = (sensibilidad + especificidad) / 2
auc = roc_auc_score(y_comprobar, arbol.predict_proba(X_comprobar)[:, 1])
siempre_no = (vn + fp) / total

print("MÉTRICAS, con el nombre que traen en los artículos (van de 0 a 1)")
metricas = pd.DataFrame([
    ["Exactitud", "accuracy", "(VP + VN) / total", exactitud, "De cada 100 hogares, en cuántos acertó"],
    ["Sensibilidad", "recall, sensitivity", "VP / (VP + FN)", sensibilidad, "De los que SÍ presentan, cuántos encontró"],
    ["Precisión", "precision", "VP / (VP + FP)", precision, "De los que señaló, cuántos sí presentan"],
    ["Especificidad", "specificity", "VN / (VN + FP)", especificidad, "De los que NO presentan, cuántos dejó en paz"],
    ["Puntaje F1", "F1-score", "media armónica de precisión y sensibilidad", f1, "Equilibrio entre encontrar y no señalar de más"],
    ["Exactitud balanceada", "balanced accuracy", "(sensibilidad + especificidad) / 2", exact_balanceada, "Exactitud justa cuando una clase es rara"],
    ["Área bajo la curva ROC", "AUC, ROC-AUC", "0.5 es azar; 1 es perfecto", auc, "Qué tan bien ordena a los hogares de más a menos riesgo"],
], columns=["Métrica", "En inglés", "Cómo se calcula", "Valor", "Qué responde"])
metricas["Valor"] = metricas["Valor"].round(2)
display(metricas)

encontrados = round(10 * sensibilidad); acertados = round(10 * precision)
print("\nLas mismas cifras, dichas en lenguaje de intervención:")
print(f"  De cada 10 hogares que SÍ presentan inseguridad alimentaria, el modelo encontró {encontrados} y se le escaparon {10 - encontrados}.")
print(f"  De cada 10 hogares que el modelo SEÑALÓ, {acertados} sí presentaban inseguridad y {10 - acertados} no.")
print(f"  En total acertó en {100 * exactitud:.0f} de cada 100 hogares. Decir siempre 'no' acertaría {100 * siempre_no:.0f}, y no serviría de nada.\n")
print("  Los que se le escaparon son hogares que un programa basado solo en el modelo dejaría fuera.")
print("  Los señalados por error son hogares a los que llegaría un apoyo que no necesitaban tanto.")

if incluir_factor_trampa:
    print("\n¡CUIDADO! Con el factor trampa el modelo parece casi perfecto. No descubrió nada:")
    print("  'un adulto dejó de comer por falta de dinero' es una de las preguntas con las que se")
    print("  construye la propia escala. El modelo solo repite la definición del resultado.")
    print("  Lección: desconfiar de un modelo que acierta demasiado.")

**Cómo leer la matriz.** Las filas son la realidad; las columnas, lo que dijo el modelo. La
diagonal son los aciertos. Los dos errores no pesan igual: un **falso negativo** es un hogar con
inseguridad que un programa dejaría fuera; un **falso positivo** es un apoyo que llega a quien no
lo necesitaba tanto. El árbol está configurado (clases balanceadas) para encontrar a los hogares
afectados aunque señale de más; por eso su sensibilidad es alta y su precisión baja.

In [ ]:
#@title 7. ¿Qué factores usa primero el árbol?
importancia = pd.Series(arbol.feature_importances_, index=X.columns).sort_values()
importancia = importancia[importancia > 0]

fig, eje = plt.subplots(figsize=(9, 0.55 * len(importancia) + 1.2))
colores = [NARANJA if v == importancia.max() else AZUL for v in importancia.values]
barras = eje.barh(importancia.index, 100 * importancia.values, color=colores)
for b, v in zip(barras, importancia.values):
    eje.text(100 * v + 1, b.get_y() + b.get_height() / 2, f"{100*v:.0f}", va="center", color=OBSCURO, fontsize=11)
eje.set_xlim(0, 100 * importancia.max() * 1.18)
eje.set_xlabel("Importancia para separar a los hogares (los factores suman 100)", color=OBSCURO, fontsize=10)
eje.set_title("Factores que más usa el árbol", color=AZUL, fontsize=13, loc="left", fontweight="bold")
for lado in ["top", "right"]: eje.spines[lado].set_visible(False)
eje.tick_params(colors=OBSCURO)
plt.tight_layout(); plt.show()

print("Que un factor aparezca primero indica que SEPARA bien a los grupos en estos datos,")
print("no que CAUSE la inseguridad alimentaria. El porqué lo explican la teoría y el campo.")

---
## El mismo problema con otros algoritmos

El árbol es el modelo más legible de la familia supervisada, no el único. La celda siguiente
entrena cuatro algoritmos más con **los mismos hogares, los mismos factores y el mismo 70/30**, y
los mide con las mismas métricas. Lo que cambia es cómo decide cada uno y si se puede leer:

- **Regresión logística:** una ecuación; un coeficiente por factor. Vive en los dos mundos, el
  estadístico y el del aprendizaje automático.
- **Bosque aleatorio** (*random forest*): cientos de árboles distintos que votan.
- **Potenciación de gradiente** (*gradient boosting*): árboles pequeños encadenados, cada uno
  corrige los errores del anterior. Es lo que suele ganar los concursos con datos tabulares.
- **Máquina de soporte vectorial** (*support vector machine*, SVM): busca la frontera que mejor
  separa a los dos grupos en el espacio de los factores.
- Las **redes neuronales** (aprendizaje profundo) quedan fuera: con siete factores y nueve mil
  hogares no tienen ventaja y necesitan muchos más datos.

In [ ]:
#@title 8. El mismo problema con otros algoritmos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

modelos = [
    ("Árbol de decisión (celda 4)", arbol, "Sí, completo: ocho perfiles"),
    ("Regresión logística", make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
        "Sí, un coeficiente por factor"),
    ("Bosque aleatorio (random forest)", RandomForestClassifier(n_estimators=300, min_samples_leaf=50, class_weight="balanced", random_state=2025),
        "No: 300 árboles que votan"),
    ("Potenciación de gradiente (gradient boosting)", HistGradientBoostingClassifier(max_depth=3, class_weight="balanced", random_state=2025),
        "No: cientos de árboles encadenados"),
    ("Máquina de soporte vectorial (SVM)", make_pipeline(StandardScaler(), SVC(class_weight="balanced", random_state=2025)),
        "No: una frontera en siete dimensiones"),
]
filas = []
for nombre, modelo, legible in modelos:
    if modelo is not arbol:
        if isinstance(modelo, Pipeline):
            modelo.fit(X_aprender, y_aprender, **{modelo.steps[-1][0] + "__sample_weight": w_aprender})
        else:
            modelo.fit(X_aprender, y_aprender, sample_weight=w_aprender)
    pred = modelo.predict(X_comprobar)
    puntaje = modelo.decision_function(X_comprobar) if hasattr(modelo, "decision_function") else modelo.predict_proba(X_comprobar)[:, 1]
    filas.append([nombre, accuracy_score(y_comprobar, pred), recall_score(y_comprobar, pred),
                  precision_score(y_comprobar, pred), f1_score(y_comprobar, pred),
                  roc_auc_score(y_comprobar, puntaje), legible])
comparacion = pd.DataFrame(filas, columns=["Modelo", "Exactitud", "Sensibilidad", "Precisión", "F1", "AUC", "¿Se puede leer?"]).round(2)
print("Los cinco con clases balanceadas, el mismo 70/30 y el mismo factor de expansión; métricas de 0 a 1 en los hogares de comprobación.\n")
display(comparacion)
print("\nLectura: con siete factores de encuesta, los modelos que no se pueden leer ganan poco o nada.")
print("Cuando ganan mucho, hay que preguntarse si algún factor ya contiene la respuesta (celda 4, casilla de la trampa).")

---
## Para pedirle ayuda al asistente de IA

El asistente explica, pero no elige la pregunta, no conoce el territorio y puede equivocarse. Todo
lo que diga se contrasta con la experiencia de campo. Copien los perfiles de la celda 5 y péguenlos
en Gemini (ícono de estrella en Colab) o en cualquier otro asistente, con esta instrucción:

> *Soy trabajadora social. Estos son los perfiles de hogar que obtuvo un árbol de decisión con la
> ENSANUT 2025 sobre inseguridad alimentaria. Descríbeme cada perfil en lenguaje sencillo, dime
> qué sí puedo concluir y qué NO puedo concluir con esto, y qué le preguntarías a alguien que
> trabaja en campo antes de usarlo para priorizar un programa.*

## Para discutir en el grupo

1. Lean en voz alta un perfil como si describieran a una familia que conocen. ¿Les suena?
2. ¿Qué factor que ustedes ven en campo no está en la base? ¿Qué pasa con los hogares que la
   encuesta no alcanza a captar?
3. El modelo "se le escaparon" algunos hogares. ¿Quiénes creen que son?
4. ¿Qué cambiaría si en vez de perfiles de población usáramos el modelo para señalar familias
   concretas con nombre y domicilio?

## Fuentes

- Instituto Nacional de Salud Pública. (2025). *Encuesta Nacional de Salud y Nutrición Continua
  2025* [Microdatos]. https://ensanut.insp.mx/encuestas/ensanutcontinua2025/
- FAO. (2012). *Escala Latinoamericana y Caribeña de Seguridad Alimentaria (ELCSA): Manual de uso
  y aplicación*.